# Notebook 4 v2 — Routing A→B avec profils utilisateur

## Objectif

Premier routing A→B réel, paramétré par **profil** (`club_road` ou `solo_casual`)
et par **intensité de préférence plaisir** (`alpha`).

## Formule utilisée

$$\text{coût}(e) = L(e) \cdot \left[1 + \alpha \cdot (\text{cost\_factor}_{\text{profil}}(e) - 1)\right]$$

- `L(e)` : longueur de l'arête en mètres
- `cost_factor_profil` : facteur calculé au Notebook 3bis (≥ 1)
- `alpha` ∈ [0, 1] : curseur utilisateur
   - 0 → routing par distance pure (cost_factor ignoré)
   - 1 → cost_factor pleinement appliqué
   - 0.5 → compromis

Avec ce formalisme :
- Une arête à `cost_factor = 1` coûte `L` (inchangé quel que soit alpha)
- Une arête à `cost_factor = 3` avec `alpha = 0.5` coûte `L × 2`
- Une arête à `cost_factor = 3` avec `alpha = 1` coûte `L × 3`

L'algo préfère donc systématiquement les "bonnes" arêtes, d'autant plus
fortement que alpha est élevé.

## Plan

1. Setup
2. Extraction du graphe (uniquement les arêtes `is_routable`)
3. Construction `networkx.MultiDiGraph`
4. Fonction de routing paramétrable
5. Tests A→B classiques
6. Comparaison profils club vs solo sur même trajet
7. Visualisation


## 1. Setup

In [22]:
import pandas as pd
import numpy as np
import networkx as nx
import time
import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine, text
from tqdm.auto import tqdm

import folium
from shapely import wkt

print(f"networkx {nx.__version__}")

networkx 3.6.1


In [23]:
DB_CONFIG = {
    "user": "postgres", "password": "4421",
    "host": "localhost", "port": 5432, "database": "velo_club",
}
url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
       f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(url, pool_pre_ping=True)

# Vérif que Notebook 3bis a bien tourné
with engine.connect() as conn:
    stats = pd.read_sql(text("""
        SELECT
            (SELECT COUNT(*) FROM osm_edges WHERE is_routable) AS routable,
            (SELECT COUNT(*) FROM edge_scores WHERE cost_factor_club > 0) AS with_cost
    """), conn)
print(stats.to_string(index=False))

 routable  with_cost
   407487     831949


## 2. Extraction du graphe

On ne charge que les arêtes `is_routable = TRUE` (exclusions dures appliquées
au Notebook 3bis). Le graphe sera d'environ 750-800k arêtes au lieu de 830k.

In [24]:
print("Extraction depuis PostGIS...")
t0 = time.time()

query = text("""
    SELECT
        e.edge_id, e.osm_way_id, e.length_m, e.highway,
        ROUND(ST_X(ST_StartPoint(e.geom))::numeric, 6) AS u_lon,
        ROUND(ST_Y(ST_StartPoint(e.geom))::numeric, 6) AS u_lat,
        ROUND(ST_X(ST_EndPoint(e.geom))::numeric, 6) AS v_lon,
        ROUND(ST_Y(ST_EndPoint(e.geom))::numeric, 6) AS v_lat,
        COALESCE(es.cost_factor_v2_club, es.cost_factor_club, 1.0) AS cost_factor_club,
        COALESCE(es.cost_factor_v2_solo, es.cost_factor_solo, 1.0) AS cost_factor_solo,
        COALESCE(es.score_club, 0)   AS score_club,
        COALESCE(es.score_final_club, 0) AS score_final_club,
        COALESCE(es.score_final_solo, 0) AS score_final_solo
    FROM osm_edges e
    LEFT JOIN edge_scores es ON es.edge_id = e.edge_id
    WHERE e.is_routable = TRUE
      AND e.geom IS NOT NULL
      AND e.length_m > 0
""")

with engine.connect() as conn:
    df_edges = pd.read_sql(query, conn)
print(f"{len(df_edges):,} arêtes en {time.time()-t0:.0f}s")

Extraction depuis PostGIS...
407,434 arêtes en 6s


In [25]:
# IDs de nœuds
df_edges["u_id"] = df_edges["u_lat"].astype(str) + "_" + df_edges["u_lon"].astype(str)
df_edges["v_id"] = df_edges["v_lat"].astype(str) + "_" + df_edges["v_lon"].astype(str)

nodes_u = df_edges[["u_id", "u_lat", "u_lon"]].rename(
    columns={"u_id": "node_id", "u_lat": "lat", "u_lon": "lon"})
nodes_v = df_edges[["v_id", "v_lat", "v_lon"]].rename(
    columns={"v_id": "node_id", "v_lat": "lat", "v_lon": "lon"})
nodes = pd.concat([nodes_u, nodes_v]).drop_duplicates("node_id").reset_index(drop=True)

print(f"Nœuds uniques : {len(nodes):,}")
print(f"Degré moyen    : {2 * len(df_edges) / len(nodes):.2f}")

Nœuds uniques : 542,407
Degré moyen    : 1.50


## 3. Construction du graphe

In [26]:
print("Construction du graphe...")
t0 = time.time()

G = nx.MultiDiGraph()

# Nœuds avec coordonnées
node_attrs = {r.node_id: {"lat": float(r.lat), "lon": float(r.lon)}
              for r in nodes.itertuples()}
G.add_nodes_from(node_attrs.items())

# Arêtes bidirectionnelles (par défaut vélo)
edge_data = []
for r in df_edges.itertuples():
    attrs = {
        "edge_id":      int(r.edge_id),
        "osm_way_id":   int(r.osm_way_id),
        "length_m":     float(r.length_m),
        "highway":      r.highway,
        "cost_factor_club": float(r.cost_factor_club),
        "cost_factor_solo": float(r.cost_factor_solo),
        "score_club":         float(r.score_club),
        "score_final_club":   float(r.score_final_club),
        "score_final_solo":   float(r.score_final_solo),
    }
    edge_data.append((r.u_id, r.v_id, attrs))
    edge_data.append((r.v_id, r.u_id, attrs))

G.add_edges_from(edge_data)
print(f"✓ Graphe construit en {time.time()-t0:.0f}s")
print(f"  Nœuds  : {G.number_of_nodes():,}")
print(f"  Arêtes : {G.number_of_edges():,}")

Construction du graphe...
✓ Graphe construit en 9s
  Nœuds  : 542,407
  Arêtes : 814,868


In [27]:
# Plus grande composante connexe (faible)
t0 = time.time()
largest_cc = max(nx.weakly_connected_components(G), key=len)
n_before = G.number_of_nodes()
G = G.subgraph(largest_cc).copy()
print(f"Plus grande CC : {G.number_of_nodes():,} / {n_before:,} "
      f"({100*G.number_of_nodes()/n_before:.1f}%) "
      f"en {time.time()-t0:.0f}s")

Plus grande CC : 90,762 / 542,407 (16.7%) en 4s


## 4. Fonction de routing

On construit une closure qui fabrique la fonction de coût selon :
- `profile` : "club_road" ou "solo_casual"
- `alpha` : intensité préférence plaisir (0 = distance, 1 = full préférence)

In [28]:
def make_cost_fn(profile: str = "club_road", alpha: float = 0.5):
    assert profile in ("club_road", "solo_casual")
    key = f"cost_factor_{profile.replace('_road','').replace('_casual','')}"
    
    def cost(u, v, d):
        # MultiDiGraph : d peut être dict de dicts
        if isinstance(d, dict) and any(isinstance(x, dict) for x in d.values()):
            return min(_edge_cost(attrs, key, alpha) for attrs in d.values())
        return _edge_cost(d, key, alpha)
    return cost


def _edge_cost(attrs, key, alpha):
    L = attrs["length_m"]
    cf = attrs[key]
    return L * (cf ** alpha)


# Test
test = {"length_m": 100, "cost_factor_club": 3.0, "cost_factor_solo": 2.0}
print(f"Arête test : 100m, cost_factor_club=3.0, cost_factor_solo=2.0")
for alpha in [0.0, 0.3, 0.5, 0.8, 1.0]:
    c = _edge_cost(test, "cost_factor_club", alpha)
    print(f"  club_road, α={alpha} → coût = {c:.0f}")

Arête test : 100m, cost_factor_club=3.0, cost_factor_solo=2.0
  club_road, α=0.0 → coût = 100
  club_road, α=0.3 → coût = 139
  club_road, α=0.5 → coût = 173
  club_road, α=0.8 → coût = 241
  club_road, α=1.0 → coût = 300


In [29]:
# Snap to nearest node
def snap_to_node(G, lat, lon):
    """Trouve le nœud le plus proche."""
    coords = np.array([(d["lat"], d["lon"]) for _, d in G.nodes(data=True)])
    ids = list(G.nodes())
    lat_rad = np.radians(lat)
    dx = (coords[:, 1] - lon) * 111320 * np.cos(lat_rad)
    dy = (coords[:, 0] - lat) * 111320
    d2 = dx*dx + dy*dy
    i = int(np.argmin(d2))
    return ids[i], float(np.sqrt(d2[i]))


def route(G, start_latlon, end_latlon, profile="club_road", alpha=0.5):
    u, du = snap_to_node(G, *start_latlon)
    v, dv = snap_to_node(G, *end_latlon)
    cost_fn = make_cost_fn(profile, alpha)
    
    try:
        path = nx.dijkstra_path(G, u, v, weight=cost_fn)
    except nx.NetworkXNoPath:
        return None
    
    total_L = 0.0
    weighted_score = 0.0
    edges = []
    coords = [(G.nodes[path[0]]["lat"], G.nodes[path[0]]["lon"])]
    score_key = f"score_final_{profile.replace('_road','').replace('_casual','')}"
    cost_key = f"cost_factor_{profile.replace('_road','').replace('_casual','')}"
    
    for i in range(len(path) - 1):
        a, b = path[i], path[i+1]
        cands = G[a][b]
        best = min(cands.keys(), key=lambda k: _edge_cost(cands[k], cost_key, alpha))
        attrs = cands[best]
        edges.append(attrs)
        total_L += attrs["length_m"]
        weighted_score += attrs["length_m"] * attrs[score_key]
        coords.append((G.nodes[b]["lat"], G.nodes[b]["lon"]))
    
    return {
        "path": path, "edges": edges, "coords": coords,
        "total_length_m": total_L,
        "mean_score": weighted_score / max(total_L, 1),
        "profile": profile, "alpha": alpha,
        "snap_m": (du, dv),
    }

print("route() prête")

route() prête


## 5. Tests A→B

In [30]:
POINTS = {
    "pantin":         (48.8922, 2.4014),
    "chevreuse":      (48.7076, 2.0387),
    "fontainebleau":  (48.4020, 2.7015),
    "cergy":          (49.0390, 2.0768),
    "rambouillet":    (48.6447, 1.8285),
    "meaux":          (48.9609, 2.8783),
}

# Test 1 : Pantin → Chevreuse, profil club, différents alpha
print("=" * 70)
print("Pantin → Chevreuse — profil club_road, variations alpha")
print("=" * 70)
results_club = {}
for a in [0.0, 0.3, 0.5, 10]:
    t0 = time.time()
    r = route(G, POINTS["pantin"], POINTS["chevreuse"], profile="club_road", alpha=a)
    if r:
        results_club[a] = r
        print(f"α = {a} | {r['total_length_m']/1000:.1f} km | "
              f"score moy {r['mean_score']:.3f} | "
              f"{time.time()-t0:.1f}s")

Pantin → Chevreuse — profil club_road, variations alpha
α = 0.0 | 53.5 km | score moy 0.428 | 0.7s
α = 0.3 | 53.7 km | score moy 0.425 | 0.6s
α = 0.5 | 54.2 km | score moy 0.444 | 0.6s
α = 10 | 79.8 km | score moy 0.428 | 2.1s


### Comparaison des profils club vs solo

Même A→B, même alpha, deux profils : on s'attend à des différences sur les
axes avec piste cyclable et les zones denses.

In [31]:
print("\n" + "=" * 70)
print("Pantin → Chevreuse — comparaison profils (alpha = 0.5)")
print("=" * 70)

r_club = route(G, POINTS["pantin"], POINTS["chevreuse"], profile="club_road", alpha=10)
r_solo = route(G, POINTS["pantin"], POINTS["chevreuse"], profile="solo_casual", alpha=10)

if r_club:
    print(f"club_road   : {r_club['total_length_m']/1000:.1f} km | "
          f"score {r_club['mean_score']:.3f}")
if r_solo:
    print(f"solo_casual : {r_solo['total_length_m']/1000:.1f} km | "
          f"score {r_solo['mean_score']:.3f}")

# Différence de chemin : nombre d'arêtes communes
if r_club and r_solo:
    ids_club = {e["edge_id"] for e in r_club["edges"]}
    ids_solo = {e["edge_id"] for e in r_solo["edges"]}
    commun = ids_club & ids_solo
    print(f"\nArêtes communes : {len(commun)} / {max(len(ids_club), len(ids_solo))}")
    print(f"Itinéraires identiques : {'oui' if ids_club == ids_solo else 'non'}")


Pantin → Chevreuse — comparaison profils (alpha = 0.5)
club_road   : 79.8 km | score 0.428
solo_casual : 67.3 km | score 0.537

Arêtes communes : 397 / 665
Itinéraires identiques : non


## 6. Visualisation

In [32]:
def render(results, start, end, zoom=10):
    center = ((start[0]+end[0])/2, (start[1]+end[1])/2)
    m = folium.Map(location=center, zoom_start=zoom, tiles="cartodbpositron")
    colors = ["blue", "green", "orange", "red", "purple"]
    for i, (label, r) in enumerate(results.items()):
        folium.PolyLine(
            r["coords"],
            color=colors[i % len(colors)],
            weight=3, opacity=0.75,
            tooltip=f"{label} | {r['total_length_m']/1000:.1f} km | score {r['mean_score']:.2f}",
        ).add_to(m)
    folium.Marker(start, tooltip="Départ", icon=folium.Icon(color="green")).add_to(m)
    folium.Marker(end, tooltip="Arrivée", icon=folium.Icon(color="red")).add_to(m)
    return m

render(results_club, POINTS["pantin"], POINTS["chevreuse"])

bizzare la rouge fait un peu nimp dans paris nan ?

In [40]:
# Échantillonne plusieurs arêtes
print("Échantillon de 10 arêtes du graphe :")
import random
sample = random.sample(list(G.edges(data=True)), 10)
for u, v, attrs in sample:
    print(f"  edge_id={attrs.get('edge_id', '?'):>8}  "
          f"highway={attrs.get('highway', '?'):15s}  "
          f"cf={attrs.get('cost_factor_club', '?'):.3f}")

# Compare distributions
import numpy as np
all_cf = [attrs["cost_factor_club"] for _, _, attrs in G.edges(data=True)]
print(f"\nDistribution cost_factor_club dans le graphe :")
print(f"  mean : {np.mean(all_cf):.3f}")
print(f"  median : {np.median(all_cf):.3f}")
print(f"  pct cf=1.0 : {sum(1 for c in all_cf if c < 1.05) / len(all_cf) * 100:.1f}%")

Échantillon de 10 arêtes du graphe :
  edge_id=  532429  highway=residential      cf=2.253
  edge_id=  714759  highway=residential      cf=1.250
  edge_id=  758113  highway=residential      cf=2.207
  edge_id=  650073  highway=residential      cf=1.791
  edge_id=  220181  highway=service          cf=4.592
  edge_id=  669025  highway=residential      cf=2.405
  edge_id=  650466  highway=residential      cf=2.261
  edge_id=  707804  highway=path             cf=2.022
  edge_id=  421146  highway=secondary        cf=1.000
  edge_id=  424131  highway=residential      cf=1.320

Distribution cost_factor_club dans le graphe :
  mean : 1.826
  median : 1.423
  pct cf=1.0 : 33.6%


en vrai ca n'a pas l'air deconnant

In [34]:
# Comparaison profils
if r_club and r_solo:
    m = render({"club_road": r_club, "solo_casual": r_solo},
               POINTS["pantin"], POINTS["chevreuse"])
    m

## 7. Benchmark

In [35]:
bench = []
pairs = [("pantin", "chevreuse"), ("pantin", "fontainebleau"),
         ("pantin", "cergy"), ("pantin", "rambouillet")]

for src, dst in pairs:
    for profile in ["club_road", "solo_casual"]:
        for a in [0.0, 0.5]:
            t0 = time.time()
            r = route(G, POINTS[src], POINTS[dst], profile=profile, alpha=a)
            if r:
                bench.append({
                    "A→B": f"{src}→{dst}",
                    "profile": profile,
                    "α": a,
                    "km": round(r["total_length_m"]/1000, 1),
                    "score": round(r["mean_score"], 3),
                    "t(s)": round(time.time()-t0, 1),
                })

bench_df = pd.DataFrame(bench)
print(bench_df.to_string(index=False))

                 A→B     profile   α    km  score  t(s)
    pantin→chevreuse   club_road 0.0  53.5  0.428   0.7
    pantin→chevreuse   club_road 0.5  54.2  0.444   0.7
    pantin→chevreuse solo_casual 0.0  53.5  0.547   0.9
    pantin→chevreuse solo_casual 0.5  53.9  0.553   0.7
pantin→fontainebleau   club_road 0.0 128.4  0.501   0.7
pantin→fontainebleau   club_road 0.5 129.0  0.504   0.7
pantin→fontainebleau solo_casual 0.0 128.4  0.569   1.0
pantin→fontainebleau solo_casual 0.5 128.8  0.572   0.7
        pantin→cergy   club_road 0.0  37.4  0.345   0.6
        pantin→cergy   club_road 0.5  38.4  0.359   0.6
        pantin→cergy solo_casual 0.0  37.4  0.436   0.6
        pantin→cergy solo_casual 0.5  37.4  0.439   0.9
  pantin→rambouillet   club_road 0.0  72.6  0.540   0.7
  pantin→rambouillet   club_road 0.5  73.2  0.551   0.7
  pantin→rambouillet solo_casual 0.0  72.6  0.634   0.7
  pantin→rambouillet solo_casual 0.5  73.0  0.638   0.7


## 8. Rapport

In [36]:
print("=" * 60)
print("RAPPORT NOTEBOOK 4 v2")
print("=" * 60)
print(f"Graphe : {G.number_of_nodes():,} nœuds / {G.number_of_edges():,} arêtes")
print(f"Exclusions dures effectives (via is_routable)")
print(f"2 profils : club_road + solo_casual")
print(f"Tests A→B : {len(set(b['A→B'] for b in bench))} paires")
print("=" * 60)

RAPPORT NOTEBOOK 4 v2
Graphe : 90,762 nœuds / 205,652 arêtes
Exclusions dures effectives (via is_routable)
2 profils : club_road + solo_casual
Tests A→B : 4 paires


## Ce qu'on a accompli

✓ Routing A→B fonctionnel avec 2 profils distincts
✓ Fonction de coût multiplicative propre (fondée Broach 2012)
✓ Paramètre α réglable par l'utilisateur
✓ Exclusion automatique des nationales / voies dangereuses
✓ Comparaison visuelle sur carte

## Limitations actuelles

- **`snap_to_node` en O(n)** : lent au premier appel, à optimiser avec KDTree
- **Pas de prise en compte du D+** : ajouté au Notebook 5 avec SRTM
- **`score_club` pas encore généralisé** : PU learning au Notebook 5
- **Sens uniques non gérés** : bidirectionnel pour tout vélo (à raffiner)

## Prochaines étapes

- **Notebook 5** : SRTM + PU learning + `cost_factor_v2` intégrant le D+ et
  le score appris plutôt qu'heuristique
